# Laser Off Code

In [ ]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

# Import

In [1]:
import time
from time import sleep, monotonic
import datetime
import numpy as np
import matplotlib.pyplot as plt
import sys
import pyvisa
import qcodes as qc
from qcodes.dataset import Measurement
from qcodes.dataset import do0d
from qcodes.dataset.experiment_container import new_experiment, load_experiment_by_name
from qcodes.dataset.plotting import plot_by_id
from qcodes.dataset.data_set import load_by_id, load_by_counter
from qcodes import initialise_or_create_database_at, new_data_set, new_experiment
from qcodes.station import Station
initialise_or_create_database_at("./2026-03-10_SNSPD4.db")
from functions import osc_set_standard, osc_check_standard, capture_trace, capture_trace_simple
from functions import snspd_dark_counts
from functions import snspd_counts_vs_wavelength
from functions import laser_set_standard, laser_get_standard
from functions import snspd_counts_vs_current
import snspd
params = snspd.snspd()

# Set up experiment
exp_name = 'SNSPD4_23_03_2026'
sample_name = '00'

try:
    exp = qc.load_experiment_by_name(exp_name, sample=sample_name)
    print('Experiment loaded. Last ID no:', exp.last_counter)
except ValueError:
    exp = new_experiment(exp_name, sample_name)
    print('Started new experiment')

Logging hadn't been started.
Activating auto-logging. Current session state plus future input saved.
Filename       : C:\Users\QNL\.qcodes\logs\command_history.log
Mode           : append
Output logging : True
Raw input log  : False
Timestamping   : True
State          : active
Qcodes Logfile : C:\Users\QNL\.qcodes\logs\260415-19244-qcodes.log
Experiment loaded. Last ID no: 486


In [ ]:
import functions
import importlib
importlib.reload(functions)
from functions import osc_set_standard, osc_check_standard, capture_trace, capture_trace_simple
from functions import snspd_dark_counts
from functions import snspd_counts_vs_wavelength
from functions import laser_set_standard, laser_get_standard

# Instruments

In [6]:
station = Station(config_file="friesland.yaml")
dmm = station.load_instrument("dmm", revive_instance=True)
yoko = station.load_instrument("yoko", revive_instance=True)
laser = station.load_instrument("laser", revive_instance=True)
MS = station.load_instrument("mso5", revive_instance=True)
pm100usb = station.load_instrument("pm100usb", revive_instance=True)
pms120 = station.load_instrument("pms120", revive_instance=True)
tc = station.load_instrument("fridge", revive_instance=True)
p_att = station.load_instrument("dmm_keithley", revive_instance=True) # excluding from snapshot because none of the parameters work anyway

# Dark Counts vs Current

Running dark counts using snspd_counts_vs_current

In [12]:
# Take same current range as ID 12
ID = 2
data = load_by_id(ID).get_parameter_data()
current = data['yoko_current']['yoko_current']
print(data['yoko_current']['yoko_current'])

[1.0e-07 2.0e-07 3.0e-07 4.0e-07 5.0e-07 6.0e-07 7.0e-07 8.0e-07 9.0e-07
 1.0e-06 1.1e-06 1.2e-06 1.3e-06 1.4e-06 1.5e-06 1.6e-06 1.7e-06 1.8e-06
 1.9e-06 2.0e-06 2.1e-06 2.2e-06 2.3e-06 2.4e-06 2.5e-06 2.6e-06 2.7e-06
 2.8e-06 2.9e-06 3.0e-06 3.1e-06 3.2e-06 3.3e-06 3.4e-06 3.5e-06 3.6e-06
 3.7e-06 3.8e-06 3.9e-06 4.0e-06]


In [13]:
current[0] - current[1]

np.float64(-1e-07)

In [14]:
np.arange(1e-7, 4.1e-6, 1e-7)

array([1.0e-07, 2.0e-07, 3.0e-07, 4.0e-07, 5.0e-07, 6.0e-07, 7.0e-07,
       8.0e-07, 9.0e-07, 1.0e-06, 1.1e-06, 1.2e-06, 1.3e-06, 1.4e-06,
       1.5e-06, 1.6e-06, 1.7e-06, 1.8e-06, 1.9e-06, 2.0e-06, 2.1e-06,
       2.2e-06, 2.3e-06, 2.4e-06, 2.5e-06, 2.6e-06, 2.7e-06, 2.8e-06,
       2.9e-06, 3.0e-06, 3.1e-06, 3.2e-06, 3.3e-06, 3.4e-06, 3.5e-06,
       3.6e-06, 3.7e-06, 3.8e-06, 3.9e-06, 4.0e-06])

In [15]:
params.device_2_name

'Line 2 Old Device'

In [16]:
from functions import set_thresholds
v_scale_old_device = 50e-3
osc_set_standard(MS, v_scale=v_scale_old_device, h_time=params.h_time_counts, h_pos=params.h_pos_counts)
threshold1, threshold2 = set_thresholds(yoko.current()) 
MS.write(f'SEARCH:SEARCH1:TRIGger:A:EDGE:THReshold {threshold1}')
MS.write(f'SEARCH:SEARCH2:TRIGger:A:EDGE:THReshold {threshold2}')

Changes compared to Measurement 4-3: device name, attenuator voltage, current range (because transition is much lower), vertical scale on scope. Current range is matched to ID 2, critical current sweep.
Thresholds pertaining to Device R7C6 are wrong for the old device, despite the current being the same. Dark counts were done for old device ID 20. 
set_thresholds function which is called by snspd_counts_vs_current has been adjusted to set the thresholds from ID 20. 

Shielding conditions: fibre wrapped and box closed

In [20]:
print(v_scale_old_device)
print(params.h_time_counts)
print(params.h_pos_counts)

currents = np.arange(1e-7, 4.1e-6, 1e-7)

# Set oscilloscope vertical and horizontal parameters 
osc_set_standard(MS, v_scale=v_scale_old_device, h_time=params.h_time_counts, h_pos=params.h_pos_counts)
osc_check_standard(MS)

# Check for clipping 
if MS.channels[0].clipping(): 
    print('Error: Clipping')

# Set attenuator voltage
v_attenuator = 4.7 # from Measurement 4-2 Notebook SNSPD4-2_Calibration.ipynb 
p_att.write(f'VOLT {v_attenuator}')

# Set standard laser parameters 
laser_set_standard(laser, 1550e-9, 7)
laser_get_standard(laser)

# Set to start current 
yoko.current(currents[0])

# Run counting without turning on laser 
snspd_counts_vs_current(MS, dmm, yoko, p_att, device_name=params.device_2_name, n_captures=10, interval=1, currents=currents, station=station)



0.05
0.1
0
MANUAL
RECORDLENGTH
625000000.0
10.0
0.01
0.0
0.05
50.0
1000000000.0
0.0
0.0
0.0
1.0
0
0.0
Power: 7.0
Frequency coarse: 193.4144THz
Wavelength (calculated) is 1550.000713493928nm
update station
Starting experimental run with id: 488. 
488
Starting current 1e-07
This acquisition will take 10s
15 35
Starting current 2e-07
This acquisition will take 10s
15 35
Starting current 3e-07
This acquisition will take 10s
15 35
Starting current 4e-07
This acquisition will take 10s
15 35
Starting current 5e-07
This acquisition will take 10s
15 36
Starting current 6e-07
This acquisition will take 10s
15 36
Starting current 7e-07
This acquisition will take 10s
15 36
Starting current 8e-07
This acquisition will take 10s
15 36
Starting current 9e-07
This acquisition will take 10s
15 37
Starting current 1e-06
This acquisition will take 10s
15 37
Starting current 1.1e-06
This acquisition will take 10s
15 37
Starting current 1.2e-06
This acquisition will take 10s
15 38
Starting current 1.3e-06
T

In [17]:
from functions import set_thresholds
v_scale_old_device = 10e-3
osc_set_standard(MS, v_scale=v_scale_old_device, h_time=params.h_time_counts, h_pos=params.h_pos_counts)
threshold1, threshold2 = set_thresholds(yoko.current()) 
MS.write(f'SEARCH:SEARCH1:TRIGger:A:EDGE:THReshold {threshold1}')
MS.write(f'SEARCH:SEARCH2:TRIGger:A:EDGE:THReshold {threshold2}')

In [25]:
############################ TURN LASER ON ############################ 
laser.enable(True)
time.sleep(10)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True


In [24]:
# Set attenuator voltage
v_attenuator = 3.6 # from Measurement 4-2 Notebook SNSPD4-2_Calibration.ipynb 
p_att.write(f'VOLT {v_attenuator}')

In [26]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: False
